<a href="https://colab.research.google.com/github/Jayadeep19/pytorch/blob/master/5.%20sequential%20models/1.%20RNN_intro_and_tensor_shape_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recurrent Neural Networks

- We will implement a Recurrent Neural Network to predict the values using the outputs from the previous time steps.

![Architecture](https://media.geeksforgeeks.org/wp-content/uploads/20260429164557969550/introduction_to_recurrent_neural_network.webp#center)

- Thus our equation for say $L_{t}$ (current output) becomes:
  - $ L_t = w_{hy}* h_t + b_y$
  - $h_t = ((w_{ih}* X_t)+b_1) + (w_{hh} * v)$
  - where,
  1. $w_{hy}, b_y$ ($w (in figure))$ is the weight matrix and bias from current hidden layer to current output
  2. $h_t$ is the current hidden state
  3.  $w_{ih}, b_1$ ($u (in figure))$ is the weight matrix and bias from the input to the hidden state
  4.  $w_{hh}, v$ is the weight matrix from previous hidden state to current hidden state and $v$ is the vector containing the information from all the previous outputs and hidden states
  - So, our current output depends on both current input and the previous outputs

  > **Note: All the weights and Biases are shared accross the layers so, the same weight will be updated during iterations.**

- ![RNN-gif](https://media4.giphy.com/media/KNCZrZEYfjtStPklsT/200.gif#center)

- This RNN architecture allows us to wrok with varying lengths of input sequences unlike the fixed length inputs for our previous Neural Networks!!
- An interesting blog written by Andrej Karpathy on [RNN](https://karpathy.github.io/2015/05/21/rnn-effectiveness/) is a must read.

![RNN](https://karpathy.github.io/assets/rnn/diags.jpeg)

- PyTorch has an implementation of [RNN](https://docs.pytorch.org/docs/2.15/generated/torch.nn.RNN.html), but here we will try to create RNN from scratch manually and also with pytorchs RNN layer.

- we will compare the execution time differences and why pytorch implementation is faster

- We will have a deeper look into the shapes and transformations of the tensors from input to output!!

## A simple RNN (manually!)

In [1]:
import torch
import torch.nn as nn
from tqdm.notebook import tqdm
import time

# Letus define a small RNN manually
class ManualRNN(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super(ManualRNN, self).__init__()

    self.input_size = input_size
    self.hidden_size = hidden_size
    self.output_size = output_size

    # define the layers (transformations from input to output)
    self.layer1 = nn.Linear(input_size, hidden_size) # the transformation from input to hidden state: contains (W_ih)
    self.layer2 = nn.Linear(hidden_size, hidden_size) # the transformation from hidden state to hidden state: contains (W_hh)
    self.layer3 = nn.Linear(hidden_size, output_size) # the transformation from hidden state to output: contains (W_hy)

  def forward(self, x, h_prev):
    '''
    x: input of shape (batch_size, input_size) # pytorch expects the first shape to be batch size
    h_prev = previous hidden state of shape (batch_size, hidden_size)
    '''

    # let us compute the current hidden state first so we can use it to calculate the current output.
    h_t = torch.tanh(self.layer1(x) + self.layer2(h_prev)) # h_t = tanh(W_ih * x + W_hh * h_prev

    y_t = self.layer3(h_t) # y_t = W_hy * h_t

    return y_t, h_t

  def init_hidden_state(self, batch_size):
    '''
    we can also add a method to initilize the hidden state for the first input.
    The v in the diagram will be initilized from zeros
    '''
    return torch.zeros(batch_size, self.hidden_size)

In [2]:
batch_size = 4   # number of sequences that are processed parallely
seq_length = 5   # Number of consequtive time steps or words in a sentence
input_size = 10   # Number of input features for eqch sequence step. Ex: n-dim vector representation for a word in sequence [or] n-features for a timestamp
hidden_size = 20    # The dimension of the hidden state representation, is a hyper parameter and can be tuned for better accuracy
output_size = 5   # It could be (no.of dimensions), (distinct categories), (values)  of the output

model1 = ManualRNN(input_size=input_size, hidden_size=hidden_size, output_size=output_size)
hidden = model1.init_hidden_state(batch_size)
print(f"The shape of the initilized hidden state tensor is {hidden.shape}")

The shape of the initilized hidden state tensor is torch.Size([4, 20])


In [3]:
#Explore the model parameters

params1 = model1.named_parameters()
print(model1)

# for key, value in params1.items():
#   print(key, value.shape)
print(params1)


ManualRNN(
  (layer1): Linear(in_features=10, out_features=20, bias=True)
  (layer2): Linear(in_features=20, out_features=20, bias=True)
  (layer3): Linear(in_features=20, out_features=5, bias=True)
)
<generator object Module.named_parameters at 0x7fb85043cc40>


In [4]:
# create random data to pass to the model to understand the dimensions cleary
start_time = time.perf_counter()

#loop over the sequence length manually!!!
predicted = []
for i in tqdm(range(seq_length), desc = 'Sequence iteration', leave = False):
  x_t = torch.randn(batch_size, input_size)
  if i == 0:
    print(f"the time step {i+1} Input shape is {x_t.shape}")
  y_t, hidden = model1(x_t, hidden)   # update the previous hidden state fro every sequence step
  predicted.append(y_t)
  print(f"the time step {i+1} output shape is {y_t.shape}, hidden state shape is {hidden.shape}")

print(f"The time taken to predict the seq length of {seq_length} by manual implementation is: {time.perf_counter() - start_time}")

Sequence iteration:   0%|          | 0/5 [00:00<?, ?it/s]

the time step 1 Input shape is torch.Size([4, 10])
the time step 1 output shape is torch.Size([4, 5]), hidden state shape is torch.Size([4, 20])
the time step 2 output shape is torch.Size([4, 5]), hidden state shape is torch.Size([4, 20])
the time step 3 output shape is torch.Size([4, 5]), hidden state shape is torch.Size([4, 20])
the time step 4 output shape is torch.Size([4, 5]), hidden state shape is torch.Size([4, 20])
the time step 5 output shape is torch.Size([4, 5]), hidden state shape is torch.Size([4, 20])
The time taken to predict the seq length of 5 by manual implementation is: 0.025762090999705833


## Pytorch built in RNN
- [nn.RNN()](https://docs.pytorch.org/docs/2.15/generated/torch.nn.RNN.html)
- In pytorch we can also define the number of layers that we want our RNN to have!!
- The argument "num_layers" in nn.RNN() is how we define if we want a single layer or stacked RNN.

![stached RNN](https://discuss.pytorch.org/uploads/default/optimized/2X/f/fb98eb0d16b722e019db59f97825aa529cb6bc08_2_685x499.png)

- IF more that one layers, we get the output only from the outermost layer. Also the **dimensions** of the input **hidden state** changes from [batch_size, hidden_size]->[no.of.directions*num_layers, batch_size, hidden_size]
  - where, no.of directions for vanilla RNN is 1 (more on directions in LSTM's)
  - num_layers by default = 1
- **Input shape** also changes depending on batch or unbatched sequence.
  - if unbatched:
    - [sequence_len, input_size] (just like our manual RNN above)
  - if batched:
    - batch_first = True:
      - [batch_size, sequence_len, input_size]
    - batch_first = False:
      - [sequence_len, batch_size, input_size]

In [5]:
class pytorchRNN(nn.Module):
  def __init__(self, input_size, hidden_size, output_size, num_layers=1):
    super(pytorchRNN, self).__init__()

    self.input_size = input_size
    self.hidden_size = hidden_size
    self.output_size = output_size
    self.num_layers = num_layers

    #define layers aka transformations in our case
    # Use the built in transformations defined in nn.RNN()
    self.rnn = nn.RNN(input_size= self.input_size,
                      hidden_size= self.hidden_size,
                      num_layers= self.num_layers,
                      batch_first=True)

    #we have to define the output layer of our RNN
    self.out_layer = nn.Linear(self.hidden_size, self.output_size)

  def forward(self, x):
    '''
    Notice how we did not pass the hidden state explicitly this is because
    pytorch handles it internally
    x : input of shape [batch_size, seq_len, input_size]
    '''

    # our rnn gives teo outputs
    # 1. rnn_out = contains the hidden state weights matrix (dim as defined in the description above)
    # 2. h_n = final hiden state for each element in the batch
    rnn_out, h_n = self.rnn(x)  #wil have dim [batch, seq_len, hidden_size] [4,5,20]

    #get the output from the last timestep
    last_output = rnn_out[:,-1,:] # now o/p shape is [batch_size, hidden_size] [4,20]

    #pass through o/p layer
    # since self.out_layer is [hidden, output_size] [20,5]
    #we get output shape is [4,20]-->[20,5] =  [batch, output]([4,5])
    output = self.out_layer(last_output)
    return output


### exploring the parameter shapes

In [6]:
# let us explore the shapes of the paramaters when num_layers = 1
 # (should be as defined in the architecture)
model2 = pytorchRNN(input_size=input_size, hidden_size=hidden_size, output_size=output_size, num_layers=1)
print(model2)

params2 = dict(model2.named_parameters())
for key, value in params2.items():
  print(key, value.shape)
#print(params1.keys())


pytorchRNN(
  (rnn): RNN(10, 20, batch_first=True)
  (out_layer): Linear(in_features=20, out_features=5, bias=True)
)
rnn.weight_ih_l0 torch.Size([20, 10])
rnn.weight_hh_l0 torch.Size([20, 20])
rnn.bias_ih_l0 torch.Size([20])
rnn.bias_hh_l0 torch.Size([20])
out_layer.weight torch.Size([5, 20])
out_layer.bias torch.Size([5])


In [7]:
# let us explore the shapes of the tensore when num_layers = 2
model3 = pytorchRNN(input_size=input_size, hidden_size=hidden_size, output_size=output_size, num_layers=2)
print(model3)

params3 = dict(model3.named_parameters())
print("parameters for numLayyers = 2")
for key, value in params3.items():
  print(key, value.shape)

pytorchRNN(
  (rnn): RNN(10, 20, num_layers=2, batch_first=True)
  (out_layer): Linear(in_features=20, out_features=5, bias=True)
)
parameters for numLayyers = 2
rnn.weight_ih_l0 torch.Size([20, 10])
rnn.weight_hh_l0 torch.Size([20, 20])
rnn.bias_ih_l0 torch.Size([20])
rnn.bias_hh_l0 torch.Size([20])
rnn.weight_ih_l1 torch.Size([20, 20])
rnn.weight_hh_l1 torch.Size([20, 20])
rnn.bias_ih_l1 torch.Size([20])
rnn.bias_hh_l1 torch.Size([20])
out_layer.weight torch.Size([5, 20])
out_layer.bias torch.Size([5])


- notice how the shape of the second layer input is "rnn.weight_ih_l1 torch.Size([20, 20])" instead of [20,10]?
  - This is beacuse the input of the second layer is the output of the previous hidden state. so it would be [hidden_size, hidden_size]

```
Input Sequence [batch_size, seq_len, input_size] (e.g., [4, 5, 10])
       │
       ▼
┌──────────────┐
│  RNN Layer 0 │ ──> Processes input_size (10) -> hidden_size (20)
└──────────────┘
       │ (Hidden state output acts as input for Layer 1)
       ▼
┌──────────────┐
│  RNN Layer 1 │ ──> Processes hidden_size (20) -> hidden_size (20)
└──────────────┘
       │
       ▼
Final RNN Output (rnn_out) [batch_size, seq_len, hidden_size] (e.g., [4, 5, 20])
```

### passing Inputs to our models

In [8]:
# Let us pass some inputs to both our models (model2, model3)
# the shape of the input changed compared to the manual RNN

# Note:No need to loop over the seq length, pytorch handles the batch of sequences parallely

start_time = time.perf_counter()
x = torch.randn(batch_size, seq_length, input_size)
output2 = model2(x)
print(f'predictions of model2 with 1 layer is {output2.shape}')

print(f"Time taken by the pytorch nn.RNN module is {time.perf_counter()-start_time}")

predictions of model2 with 1 layer is torch.Size([4, 5])
Time taken by the pytorch nn.RNN module is 0.0032697950009605847


In [9]:
#for model 3  with 2 layers
# the input dimensions doesnt need to be changed even for n num layers
x = torch.randn(batch_size, seq_length, input_size)
output3 = model3(x)
print(f'predictions of model3 with 2 layers is {output3.shape}')

predictions of model3 with 2 layers is torch.Size([4, 5])


- same output shape with different layers

- Notice the speed difference between the manual (for loop) nd pytorch implementation (coz of parallel computing)
>- **0.02185 seconds (manual) vs 0.00196 seconds (parallel computing)**
>- **pytorch's built in implementation is approximately 11 times faster!!!**